In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not schema:
    raise ValueError(
        "Inserire nel widget 'schema' il nome dello schema da validare"
    )

base = f"{catalog}.{schema}"

print(f"Schema sottoposto a controllo: {base}")

In [0]:
tables = [
    "bronze_breweries",
    "bronze_breweries_job_config",
    "cdc_breweries_events",
    "silver_staging_breweries",
    "silver_breweries",
    "gold_breweries",
    "agg_breweries",
]

for table in tables:
    full_name = f"{base}.{table}"

    try:
        count_rows = spark.table(full_name).count()
        print(f"Table {full_name} exists with {count_rows} rows.")
    except Exception as e:
        print(f"Table {full_name} does not exist or cannot be accessed. Error: {e}")

***
**Verificare current_page**

In [0]:
display(
    spark.table(f"{base}.bronze_breweries_job_config")
)

***
**Verificare lo storico SCD2**

In [0]:
storico = (
    spark.table(f"{base}.gold_breweries")
    .select(
        "id",
        "name",
        "phone",
        "street",
        "__START_AT",
        "__END_AT",
    )
    .orderBy("id", "__START_AT")
)

display(storico)

***
**Breweries Version**

In [0]:
from pyspark.sql import functions as F

gold_df = spark.table(f"{base}.gold_breweries")

display(
    gold_df
    .groupBy("id", "name")
    .agg(
        F.count("*").alias("number_versions")
    )
    .where(F.col("number_versions") > 1)
    .orderBy(F.col("number_versions").desc())
)

In [0]:
display(
    gold_df
    .where(F.col("__END_AT").isNull())
)

In [0]:
display(
    gold_df
    .where(F.col("__END_AT").isNull())
    .groupBy("id")
    .count()
    .where(F.col("count") != 1)
)

***
**Verify agg_breweries**

In [0]:
display(
    spark.table(f"{base}.agg_breweries")
    .orderBy(F.col("num_breweries").desc())
)